# Variant-to-function evidence agent: Colab runner

Thin front end over the `variant_evidence` package. All logic lives in the repo; this notebook only
fetches data and calls it. **Three pieces are unverified until you run this** (they were written without
network access): the AG2 download format, the Open Targets query, and the held-out gold loci.
Where a cell can fail on that, the error message tells you what to paste back.

Runtime: CPU is enough.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/variant-evidence-data'
import os; os.makedirs(DATA_DIR, exist_ok=True)

REPO = 'https://github.com/Raj4884/Variant-to-function-evidence-agent.git'
if os.path.exists('/content/repo'):
    !git -C /content/repo pull -q
else:
    !git clone -q $REPO /content/repo
%cd /content/repo
!pip install -q -e ".[dev,llm]" duckdb pandas
!python -m pytest -q

## 2. Get the AlphaGenome 2 scores
Download from https://deepmind.google.com/science/alphagenome/downloads (check the terms: non-commercial),
then upload the file(s) into the Drive folder `variant-evidence-data` created above.
Do not download inside Colab if the page needs a browser login.

In [ ]:
import glob
files = sorted(glob.glob(f'{DATA_DIR}/**/*', recursive=True))
for f in files: print(f, os.path.getsize(f) if os.path.isfile(f) else '')

## 3. Inspect the raw file (edit `RAW`)

In [ ]:
import duckdb
RAW = f'{DATA_DIR}/PUT_FILE_NAME_HERE'   # .parquet, .csv, .csv.gz, .tsv ...
reader = 'read_parquet' if RAW.endswith('.parquet') else 'read_csv_auto'
con = duckdb.connect()
print(con.execute(f"DESCRIBE SELECT * FROM {reader}('{RAW}')").fetchdf())
print(con.execute(f"SELECT * FROM {reader}('{RAW}') LIMIT 5").fetchdf().T)

## 4. Map columns and normalise
Fill `RENAME` (normalised name -> raw column name) from the output above. If the raw file has no rsID column,
filter by chromosome/position/alleles instead by editing the `WHERE` clause. **Check the effect units**:
if the raw effect is log2 fold change, set `LFC_SCALE = math.log(2)`.

In [ ]:
import math
from variant_evidence.convert import normalise_rows, write_normalised_csv

RSIDS = ['rs2836882','rs12913832','rs6497271','rs1175550','rs6673829','rs2038740',
         'rs13066062','rs35705950','rs3820282','rs1421085','rs12740374']
RENAME = {'variant': '???', 'gene': '???', 'condition': '???', 'lfc': '???', 'log_cpm': '???'}
VARIANT_COL = RENAME['variant']
LFC_SCALE = 1.0

in_list = ','.join(f"'{r}'" for r in RSIDS)
cur = con.execute(f"SELECT * FROM {reader}('{RAW}') WHERE \"{VARIANT_COL}\" IN ({in_list})")
cols = [d[0] for d in cur.description]
rows = (dict(zip(cols, r)) for r in cur.fetchall())
n = write_normalised_csv(normalise_rows(rows, RENAME, rsids=set(RSIDS), lfc_scale=LFC_SCALE),
                         f'{DATA_DIR}/ag2_normalised.csv')
print(n, 'rows written')

In [ ]:
import pandas as pd
df = pd.read_csv(f'{DATA_DIR}/ag2_normalised.csv')
print(df.groupby('variant').size())
print(df.lfc.describe())   # sanity: effects should mostly sit within about +-1; paper call threshold is |LFC|>0.1

## 5. Open Targets (live)
Step 1 lists the real field names. If the query in step 2 errors, fix `CREDIBLE_SET_QUERY` and
`rows_from_response` in `src/variant_evidence/providers/open_targets.py` (or paste the output to Claude).

In [ ]:
from variant_evidence.providers import open_targets as ot
for t in ['Variant', 'CredibleSet']:
    try: print(t, ot.introspect(t))
    except Exception as e: print(t, 'ERROR', e)

In [ ]:
print(ot.resolve_variant_id('rs12740374'))
from variant_evidence.models import VariantQuery
try:
    print(ot.http_transport()(VariantQuery(rsid='rs12740374'))[:3])
except Exception as e:
    print('ERROR:', e)

## 6. Run the agent

In [ ]:
from variant_evidence.agent import run_agent, llm_synthesizer
from variant_evidence.models import VariantQuery
from variant_evidence.providers.ag2_precomputed import AG2PrecomputedProvider

USE_OT = False      # set True once step 5 works
USE_LLM = False     # set True to use Claude for wording (key in Colab Secrets as ANTHROPIC_API_KEY)
providers = [AG2PrecomputedProvider(f'{DATA_DIR}/ag2_normalised.csv')]
if USE_OT: providers.append(ot.OpenTargetsProvider(ot.http_transport()))
synth = None
if USE_LLM:
    from google.colab import userdata; import anthropic
    synth = llm_synthesizer(anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY')), 'claude-sonnet-5-5')

report = run_agent(VariantQuery(rsid='rs12913832'), providers, synth)
print(report.model_dump_json(indent=2)[:4000])

## 7. Evaluation on the gold loci

In [ ]:
!python eval/run_eval.py --ag2 {DATA_DIR}/ag2_normalised.csv

## 8. What to send back
Paste into the chat: (a) the file name and the DESCRIBE + first rows from step 3, (b) any step 5 errors or the
introspection output, (c) the eval summary lines from step 7. The two `heldout_candidate` loci came from memory
and must be checked against the literature before any number is quoted.